# Comparação das seis representações com XGBoost — testes de robustez

Extensão do código simples original. Os textos e os rótulos foram preservados. A versão do Fake.Br usa os grupos reforçados registrados no pacote. O notebook permite conferir uma condição de treino/teste e, opcionalmente, executar os procedimentos completos dos arquivos de apoio. Os resultados publicados foram obtidos localmente em CPU; os vetores congelados vieram das extrações originais no Colab.

Extraia os quatro arquivos de reprodução identificados em LEIA_ME_REPRODUCAO.md na mesma pasta de trabalho. O pacote contém dados, grupos, vetores congelados, critérios de seleção e apoios. Não são necessários pesos dos encoders nem nova extração. Para manter o andamento no Colab, use uma pasta persistente já autorizada no Drive; a montagem não é feita automaticamente por este notebook.

In [ ]:
%pip install -q numpy==2.2.6 scipy==1.15.3 pandas==2.2.3 scikit-learn==1.6.1 gensim==4.4.0 xgboost==3.0.2 datasketch==1.6.5 psutil==7.0.0

## 1. Local dos arquivos e condição escolhida

Informe a pasta que contém `robustez_privada_2026_11_06` e `experimentos_divisoes_2026_11_05`. No Colab, o caminho padrão abaixo pressupõe que o pacote foi extraído em `/content`. O exemplo usa FakeTrue.Br e semente42; as cinco sementes do estudo são42,43,44,45,46. A semente do treinamento permanece42 para comparar a mudança das notícias separadas.

In [ ]:
from pathlib import Path
import os, sys, json, time
PASTA = Path(os.environ.get('TG_RAIZ', '/content/TG_REPRODUCAO_ROBUSTEZ'))
BASE = 'faketrue'
SEMENTE_DIVISAO = 42
EXECUTAR_ESTUDO_COMPLETO = False
os.environ['TG_RAIZ'] = str(PASTA)
import subprocess
CODIGO = Path.cwd()
if not (CODIGO / 'protocolo_robustez_20261106.py').is_file():
    CODIGO = Path('/content/tg-desinformacao')
    if not CODIGO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/diego-henrique-vaz/avaliacao-de-representacoes-vetoriais-na-identificacao-de-desinformacao', str(CODIGO)], check=True)
os.environ.pop('TG_WORKER', None)
APOIOS = CODIGO
assert APOIOS.is_dir(), 'Extraia o pacote e confira PASTA.'
sys.path.insert(0, str(APOIOS))
SAIDA = PASTA / 'robustez_privada_2026_11_06' / 'validacao_notebook' / f'{BASE}_seed{SEMENTE_DIVISAO}'
SAIDA.mkdir(parents=True, exist_ok=True)
print('Saída:', SAIDA)

## 2. Bibliotecas e documentos

As versões são conferidas antes de usar os arquivos. A função de leitura aplica apenas o mapa de grupos do Fake.Br; não muda seu texto ou classe. Notícias relacionadas não podem ficar em lados diferentes da divisão.

In [ ]:
import numpy as np
import pandas as pd
import importlib.metadata as im
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from protocolo_robustez_20261106 import load_data
from executar_robustez_20261106 import traditional, PARAMS, METHODS, E
from executar_divisoes_20261105 import sha
VERSOES = {p: im.version(p) for p in ['numpy', 'scipy', 'pandas', 'scikit-learn', 'gensim', 'xgboost']}
assert VERSOES == {'numpy':'2.2.6','scipy':'1.15.3','pandas':'2.2.3','scikit-learn':'1.6.1','gensim':'4.4.0','xgboost':'3.0.2'}, VERSOES
entradas = json.loads((PASTA / 'robustez_privada_2026_11_06/dados/ENTRADAS_FIXAS.json').read_text(encoding='utf8'))
assert sha(PASTA / 'robustez_privada_2026_11_06/dados' / f'{BASE}.csv') == entradas['csv_sha256'][BASE]
if BASE == 'fakebr':
    assert sha(PASTA / 'robustez_privada_2026_11_06/dados/GRUPOS_REFORCADOS_FAKEBR.csv') == entradas['groups_map_sha256']
conferencia_caches = json.loads((E / 'evidencias/VALIDACAO_CACHES_COMPLETA.json').read_text(encoding='utf8'))
dados = load_data(BASE)
print('Documentos:', len(dados), '| Grupos:', dados.group_id.nunique())
print('Classes: verdadeira=0 e falsa=1')

In [ ]:
treino, teste = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMENTE_DIVISAO).split(dados, groups=dados.group_id))
assert not set(dados.iloc[treino].group_id) & set(dados.iloc[teste].group_id)
assert not set(dados.iloc[treino].canonical_sha256) & set(dados.iloc[teste].canonical_sha256)
assert dados.iloc[treino].label.nunique() == dados.iloc[teste].label.nunique() == 2
particoes = dados[['doc_id','group_id','canonical_sha256','label']].copy()
particoes['split'] = 'train'
particoes.loc[teste, 'split'] = 'test'
particoes.to_csv(SAIDA / 'particoes.csv', index=False)
print('Treino:', len(treino), '| Teste:', len(teste))

## 3. Representação, treinamento e avaliação

O procedimento é o mesmo para cada técnica: obter os vetores, treinar o XGBoost e classificar as notícias reservadas. TF-IDF, Word2Vec e FastText aprendem somente com o treino escolhido. Nos três modelos pré-treinados, os vetores já estão salvos; basta selecionar as linhas corretas. O classificador e seu limiar0,5 permanecem iguais. Os apoios contêm os parâmetros e as verificações extensas para manter este notebook próximo da organização do código inicial.

In [ ]:
resultados = []
for metodo in METHODS:
    inicio = time.perf_counter()
    if metodo in ['tfidf','word2vec','fasttext']:
        X_treino, X_teste, _, detalhes = traditional(dados, treino, teste, metodo, {})
    else:
        arquivo_vetores = E / 'vetores_contextuais' / BASE / metodo / 'all.npy'
        conferencia = next(r for r in conferencia_caches if r['base'] == BASE and r['method'] == metodo)
        assert sha(arquivo_vetores) == conferencia['all_sha256']
        vetores = np.load(arquivo_vetores, mmap_mode='r', allow_pickle=False)
        assert len(vetores) == len(dados) and np.isfinite(vetores).all()
        X_treino, X_teste = vetores[treino], vetores[teste]
    classificador = XGBClassifier(**PARAMS)
    classificador.fit(X_treino, dados.iloc[treino].label.to_numpy())
    probabilidades = classificador.predict_proba(X_teste)[:, 1]
    predicoes = (probabilidades >= 0.5).astype(int)
    tabela = dados.iloc[teste][['doc_id','group_id','label']].copy()
    tabela['prob_fake'] = probabilidades
    tabela['prediction'] = predicoes
    tabela.to_csv(SAIDA / f'{metodo}_predicoes.csv', index=False)
    classificador.save_model(SAIDA / f'{metodo}_xgboost.json')
    medidas = {'metodo':metodo, 'macro_f1':float(f1_score(tabela.label, predicoes, average='macro')),
               'acuracia':float(accuracy_score(tabela.label, predicoes)), 'segundos_locais':time.perf_counter()-inicio}
    relatorio = classification_report(tabela.label, predicoes, labels=[0,1], target_names=['Verdadeira','Falsa'], output_dict=True, zero_division=0)
    (SAIDA / f'{metodo}_metricas.json').write_text(json.dumps({'medidas':medidas, 'por_classe':relatorio,
        'matriz_confusao':confusion_matrix(tabela.label, predicoes, labels=[0,1]).tolist()}, ensure_ascii=False, indent=2), encoding='utf8')
    resultados.append(medidas)
    print(metodo, '| macro-F1:', round(medidas['macro_f1'], 4))
    del classificador, X_treino, X_teste
pd.DataFrame(resultados).to_csv(SAIDA / 'resultados.csv', index=False)

## 4. Conferir a tabela

Macro-F1 é a média do F1 das duas classes, com o mesmo peso para ambas. Este exemplo contém um sorteio; não confundir seu resultado com a média de cinco. O desvio padrão do estudo descreve a dispersão dos escores entre sorteios da mesma base, sem média entre bases e sem assumir independência entre testes sobrepostos.

In [ ]:
print(pd.DataFrame(resultados).to_string(index=False))
print('Arquivos preservados em:', SAIDA)

## 5. Execução completa opcional

Mude EXECUTAR_ESTUDO_COMPLETO para True somente se quiser refazer ou retomar todo o estudo. O executor salva cada método e confere assinaturas/hashes ao retomar. O apoio aplica os critérios congelados de comprimento e fonte; a transferência reutiliza modelos e transformações aprendidos na base de origem, sem aprender no alvo. Os caches originais e as seleções auditadas precisam estar no pacote. O teste de fontes só foi viável no PT-EU, com desequilíbrio de classes documentado. Os tempos dependem do ambiente e não são prometidos antecipadamente.

In [ ]:
if EXECUTAR_ESTUDO_COMPLETO:
    from executar_robustez_20261106 import run
    from executar_controles_20261106 import run_controls, run_transfer
    run()
    run_controls()
    run_transfer()
    print('Execução completa encerrada; conferir os relatórios de validação antes de atualizar o artigo.')
else:
    print('Somente a condição de exemplo foi executada.')

## Recuperação e limites

Preservar o primeiro backup histórico e os três snapshots completos mais recentes, validados antes de enviar qualquer excedente à Lixeira. Resultados científicos e dados ativos não entram na exclusão rotativa. A montagem opcional do Drive depende da conta do usuário e deve apontar para uma pasta específica do TG. As repetições não corrigem automaticamente vieses de fonte, gênero ou rótulo; os controles e a transferência são avaliações complementares. Não selecionar a melhor semente ou configuração pelo resultado do teste. Os 72 resultados anteriores permanecem históricos; no Fake.Br, os grupos reforçados mudam a divisão e impedem tratar o novo seed42 como o mesmo teste antigo.